<a href="https://colab.research.google.com/github/whitestones011/deep_learning/blob/colab/finetunning_sentence_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FINE-TUNNING SENTENCE CLASSIFIER

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
!pip install datasets -qU

from datasets import load_dataset

# Load pre-trained BERT transformer model

In [ ]:
model_id = "bert-base-uncased"

BERT is a bidirectional transformer pretrained on unlabeled text to predict masked tokens in a sentence and to predict whether one sentence follows another. The main idea is that by randomly masking some tokens, the model can train on text to the left and right, giving it a more thorough understanding.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

In [ ]:
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
    "The black cat on the roof"
]

In [ ]:
# training
inputs = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")
print(inputs)

In [ ]:
print(tokenizer.decode(inputs['input_ids'][0]).split())

In [ ]:
len(inputs['input_ids'][0])

In [ ]:
model(**inputs).logits.softmax(dim=-1)

In [ ]:
embeddings = model.bert.embeddings(inputs['input_ids'])

In [ ]:
embeddings.shape

In [ ]:
# CLS token - first sentence
embeddings[0,0,:10]

In [ ]:
# CLS token - second sentence
embeddings[1,0,:10]

Two sentences have CLS token at the begging of the sentense.

In [ ]:
import torch

In [ ]:
# add labels
inputs["labels"] = torch.tensor([1, 1, 0])

In [ ]:
# define optimizer
optimizer = torch.optim.AdamW(model.parameters())

In [ ]:
# train model
loss = model(**inputs).loss
loss.backward()
optimizer.step()

In [ ]:
with torch.no_grad():
    logits = model(**inputs).logits

In [ ]:
logits.softmax(dim=-1).argmax(dim=-1)

In [ ]:
embeddings_after_training = model.bert.embeddings(inputs['input_ids'])

In [ ]:
# CLS token - first sentence
embeddings_after_training[0,0,:10]

In [ ]:
# CLS token - second sentence
embeddings_after_training[1,0,:10]

# LOAD DATASET

In [ ]:
# LOAD DATASET
raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

In [ ]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

In [ ]:
raw_train_dataset.features

Tokenize inputs

In [ ]:
raw_train_dataset[0]

In [ ]:
tokenizer

In [ ]:
inputs = tokenizer(raw_train_dataset[0]["sentence1"], raw_train_dataset[0]["sentence2"], truncation=True, padding=True)
inputs

In [ ]:
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"]))

In [ ]:
def tokenize_func(x):
  return tokenizer(x["sentence1"], x["sentence2"], truncation=True)

In [ ]:
tokenized_dataset = raw_datasets.map(tokenize_func, batched=True)
tokenized_dataset

In [ ]:
# DYNAMIC PADDING
from transformers import DataCollatorWithPadding

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# ADDITIONAL STEPS
#  Remove the columns corresponding to values the model does not expect (like the sentence1 and sentence2 columns).
#  Rename the column label to labels (because the model expects the argument to be named labels).
#  Set the format of the datasets so they return PyTorch tensors instead of lists.

In [ ]:
remove_cols = ['sentence1', 'sentence2', 'idx',]

In [ ]:
tokenized_dataset = (
    tokenized_dataset
    .remove_columns(remove_cols)
    .rename_column("label", "labels")
    .with_format("torch")
)

In [ ]:
samples = tokenized_dataset['train'][:8]
# length of the inputs
[len(x) for x in samples["input_ids"]]

When we train model, the input should be a matrix - meaning all inputs of the same length.

So all inputs in the batch should have same length. We can achieve this by PADing the short inputs.

However, by PADing the model has extra PAD tokens - the model predictions would be different on padded input compared to unpadded.

To avoid this, we use ATTENTION_MASK to tell model where the PADded is.

This is done under the scene by using 'padding=True' flag in the tokenizer.

In [ ]:
tokenizer.pad_token_id

In [ ]:
tokenizer.convert_ids_to_tokens(0)

# TRAINER API

In [ ]:
from transformers import TrainingArguments
from transformers import Trainer

In [ ]:
# Define our model as AutoModelForSequenceClassification class, with two labels
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

In [ ]:
!pip install evaluate -qU
import evaluate

def compute_metrics(eval_pred):
  metric = evaluate.load("glue", "mrpc")
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Define trainer
training_args = TrainingArguments("test-trainer", eval_strategy="epoch")

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# trainer.train()

# Pytorch training loop



*   get batch of the training data and feed into the model
*   with labels compute the loss
*   calculate the gradients of the model as the deravative of the loss with respect to each model weight
* those gradients are used by optimizer to update weights



In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Create a DataLoader to convert the dataset into the batches
train_loader = DataLoader(tokenized_dataset["train"], shuffle=True, batch_size=8, collate_fn=data_collator)
eval_loader = DataLoader(tokenized_dataset["validation"], batch_size=8, collate_fn=data_collator)

In [ ]:
# check
for batch in train_loader:
  break
{k: v.shape for k, v in batch.items()}


In [ ]:
# initiate model
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

In [ ]:
# check on one batch
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

In [ ]:
# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [ ]:
# learning rate decay - scheduler decay from 5e-5 to 0
from transformers import get_scheduler

num_epoch = 1
num_training_steps = num_epoch * len(train_loader)
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)
num_training_steps

In [ ]:
# training loop

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

In [ ]:
batch.keys()

In [ ]:
!pip install evaluate -qU

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
%%time
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()

counter = 0
for epoch in range(num_epoch):
  for batch in train_loader:
    if counter == 10:
        break
    batch = {k: v.to(device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    progress_bar.update(1)
    counter += 1

In [ ]:
%%time
model.eval()
for batch in eval_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    # Disable gradient computation and reduce memory consumption.
    with torch.no_grad():
      outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    accuracy.add_batch(predictions=predictions, references=batch['labels'])

accuracy.compute()

# ACCELERATE

In [ ]:
from accelerate import Accelerator
from transformers import default_data_collator

accelerator = Accelerator()

train_loader, eval_loader, model, optimizer = accelerator.prepare(
    train_loader, eval_loader, model, optimizer
)

In [ ]:
# !accelerate config

In [ ]:
# DISTRIBUTED TRAINING
def training_func():
  progress_bar = tqdm(range(num_training_steps))

  model.train()

  counter = 0
  for epoch in range(num_epoch):
    for batch in train_loader:
      if counter == 10:
          break
      # batch = {k: v.to(device) for k, v in batch.items()}
      outputs = model(**batch)
      loss = outputs.loss
      # loss.backward()
      accelerator.backward(loss)

      optimizer.step()
      lr_scheduler.step()
      optimizer.zero_grad()
      progress_bar.update(1)
      counter += 1

In [ ]:
# %%time
# from accelerate import notebook_launcher

# notebook_launcher(training_func)

In [ ]:
%%time
training_func()

In [ ]:
# DISTRIBUTED EVALUATION

metric = evaluate.load("glue", "mrpc")
model.eval()

for batch in eval_loader:
    # Disable gradient computation and reduce memory consumption.
    with torch.no_grad():
      outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(
        predictions=accelerator.gather(predictions),
        references=accelerator.gather(batch['labels'])
        )

metric.compute()